# Phase 2: Resumable `Qwen/Qwen3-Reranker-8B` Teacher Scoring (Colab / Gemini Ultra)

Self-contained, resumable notebook for scoring unscored Parquet shards from `christianengel/rerank-mini-distill-shards` (`track_b_unscored/`) using `Qwen/Qwen3-Reranker-8B` (`yes_logit - no_logit`) and streaming completed shards back to `track_b_scored/` after every shard.

- Recommended Colab runtime: **A100 40GB/80GB**, **H100**, or **L4** (BF16).
- Safe to disconnect/restart at any time: already-uploaded shards in `track_b_scored/` are automatically skipped.

In [ ]:
!pip -q install -U 'transformers>=4.51' 'huggingface_hub>=0.30' pyarrow tqdm accelerate

In [ ]:
import gc, os, time
from pathlib import Path
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Set HF_TOKEN in Colab Secrets (preferred) or os.environ['HF_TOKEN'].
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN in Colab Secrets or os.environ'

HF_DATASET_REPO = 'christianengel/rerank-mini-distill-shards'
UNSCORED_PREFIX = 'track_b_unscored'
SCORED_PREFIX = 'track_b_scored'
SHARD_START = 0   # Adjust to split work across multiple Colab sessions or alongside Vast.ai
SHARD_END = -1    # Exclusive (-1 = all remaining)
MAX_LENGTH = 6144
BASE_BATCH_SIZE = 16

print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
TEACHER_ID = 'Qwen/Qwen3-Reranker-8B'
PREFIX = (
    '<|im_start|>system\n'
    'Judge whether the Document meets the requirements based on the Query and the Instruct provided. '
    'Note that the answer can only be "yes" or "no".<|im_end|>\n'
    '<|im_start|>user\n'
)
SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID, padding_side='left')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, dtype=torch.bfloat16, attn_implementation='sdpa', device_map='auto'
).eval()

TOKEN_YES = int(tokenizer.convert_tokens_to_ids('yes'))
TOKEN_NO = int(tokenizer.convert_tokens_to_ids('no'))
PREFIX_IDS = tokenizer.encode(PREFIX, add_special_tokens=False)
SUFFIX_IDS = tokenizer.encode(SUFFIX, add_special_tokens=False)
BODY_BUDGET = MAX_LENGTH - len(PREFIX_IDS) - len(SUFFIX_IDS)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)

@torch.inference_mode()
def score_texts(texts, base_bs=BASE_BATCH_SIZE):
    bodies = tokenizer(
        texts, padding=False, truncation='longest_first',
        return_attention_mask=False, max_length=BODY_BUDGET, add_special_tokens=False
    )['input_ids']
    full_ids = [PREFIX_IDS + b + SUFFIX_IDS for b in bodies]
    order = sorted(range(len(full_ids)), key=lambda i: len(full_ids[i]), reverse=True)
    out = np.empty(len(full_ids), dtype=np.float32)
    device = next(model.parameters()).device
    vscale = max(1.0, VRAM_GB / 24.0)
    idx = 0
    pbar = tqdm(total=len(order))
    while idx < len(order):
        L = len(full_ids[order[idx]])
        factor = 1.0 if L <= 512 else (0.5 if L <= 1024 else (0.25 if L <= 2048 else (0.12 if L <= 4096 else 0.06)))
        bs = max(1, int(base_bs * min(vscale, 4.0) * factor))
        chunk = order[idx:idx+bs]
        padded = tokenizer.pad({'input_ids': [full_ids[i] for i in chunk]}, padding=True, return_tensors='pt')
        padded = {k: v.to(device, non_blocking=True) for k, v in padded.items()}
        try:
            logits = model(**padded).logits[:, -1, :]
            vals = (logits[:, TOKEN_YES] - logits[:, TOKEN_NO]).float().cpu().numpy()
            for p, orig_i in enumerate(chunk):
                out[orig_i] = vals[p]
            idx += len(chunk)
            pbar.update(len(chunk))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            base_bs = max(1, base_bs // 2)
    pbar.close()
    return out

In [ ]:
api = HfApi(token=HF_TOKEN)
files = api.list_repo_files(HF_DATASET_REPO, repo_type='dataset')
unscored = sorted(f for f in files if f.startswith(UNSCORED_PREFIX + '/') and f.endswith('.parquet'))
scored_set = {Path(f).name for f in files if f.startswith(SCORED_PREFIX + '/') and f.endswith('.parquet')}
target = unscored[SHARD_START:] if SHARD_END < 0 else unscored[SHARD_START:SHARD_END]
print(f'Total unscored={len(unscored)}, already scored={len(scored_set)}, targeting={len(target)}')

work_dir = Path('/content/scored_shards')
work_dir.mkdir(parents=True, exist_ok=True)

for remote_in in target:
    fname = Path(remote_in).name
    if fname in scored_set:
        print(f'[skip] {fname} already uploaded')
        continue
    local_in = Path(hf_hub_download(HF_DATASET_REPO, remote_in, repo_type='dataset', token=HF_TOKEN))
    rows = pq.read_table(local_in).to_pylist()
    texts, offsets = [], []
    for r in rows:
        inst = r.get('instruction') or 'Given a web search query, retrieve relevant passages that answer the query'
        s = len(texts)
        for d in r['docs']:
            texts.append(f'<Instruct>: {inst}\n<Query>: {r["query"]}\n<Document>: {d}')
        offsets.append((s, len(texts)))
    t0 = time.perf_counter()
    scores = score_texts(texts)
    out_rows = []
    for r, (s, e) in zip(rows, offsets):
        raw = [float(x) for x in scores[s:e]]
        m = max(raw)
        out_rows.append({
            'query_id': r['query_id'], 'source': r['source'], 'domain': r['domain'],
            'license': r['license'], 'query': r['query'], 'docs': r['docs'],
            'doc_labels': r['doc_labels'], 'teacher_logits': raw,
            'teacher_logits_shifted': [x - m for x in raw],
        })
    local_out = work_dir / fname
    pq.write_table(pa.Table.from_pylist(out_rows), local_out, compression='zstd')
    api.upload_file(
        path_or_fileobj=str(local_out),
        path_in_repo=f'{SCORED_PREFIX}/{fname}',
        repo_id=HF_DATASET_REPO,
        repo_type='dataset',
    )
    print(f'[done] {fname}: {len(texts)} pairs in {time.perf_counter()-t0:.1f}s -> uploaded')